In [1]:
import pandas as pd
import numpy as np



In [2]:
housing = pd.read_csv('processed_data/02_data.csv')

In [3]:
housing.columns

Index(['城市', '区域', '板块', 'price', 'unit_price', '看房时间', '房屋户型', '所在楼层', '建筑面积',
       '房屋朝向', '建筑结构', '装修情况', '梯户比例', '配备电梯', '别墅类型', '挂牌时间', '交易权属', '上次交易',
       '房屋用途', '房屋年限', '产权所属', '房源标签', '核心卖点', '户型介绍', '周边配套', '交通出行', 'x',
       'y', '年份', 'nearest_park_dist_km', 'nearest_park_index',
       'nearest_park_industry', 'log_price', 'log_unit_price', 'area',
       'distance_to_center'],
      dtype='object')

In [4]:
import pandas as pd
import re

def extract_counts(text):
    # 处理空值
    if pd.isna(text):
        return pd.Series({'rooms': 0, 'halls': 0, 'kitchens': 0, 'bathrooms': 0})
    
    # 转换为字符串确保安全
    text = str(text)
    
    # 定义提取逻辑的辅助函数：找到返回数字，找不到返回0
    def get_num(pattern, string):
        match = re.search(pattern, string)
        return int(match.group(1)) if match else 0

    # 提取各项数据
    res = {
        'rooms': get_num(r'(\d+)室', text),
        'halls': get_num(r'(\d+)厅', text),
        'kitchens': get_num(r'(\d+)厨', text),
        'bathrooms': get_num(r'(\d+)卫', text)
    }
    
    return pd.Series(res)

new_cols = housing['房屋户型'].apply(extract_counts)
housing = pd.concat([housing, new_cols], axis=1)
housing = housing.drop('房屋户型', axis=1)

In [5]:
from utils import multi_label_explosion
housing = multi_label_explosion(housing, '看房时间',' ')
housing = housing.drop('看房时间', axis=1)

检测到的基础看房时间类型共有 4 种：
['只周末可看' '下班后可看' '提前预约随时可看' '有租户需预约']


In [6]:
def extract_floor(text):
    # 处理空值
    if pd.isna(text):
        return pd.Series({'floor_level': None, 'total_floors': 0})
    
    text = str(text)
    # 正则解释：
    # ^(.+?)  -> 匹配开头的文字（楼层位置）
    # \s* -> 匹配可能存在的空格
    # \(共(\d+)层\) -> 匹配 (共数字层)
    match = re.search(r'^(.+?)\s*\(共(\d+)层\)', text)
    
    if match:
        return pd.Series({
            'floor_level': match.group(1),        # 提取：中楼层
            'total_floors': int(match.group(2))   # 提取：6
        })
    else:
        # 如果格式不匹配，返回空值
        return pd.Series({'floor_level': None, 'total_floors': 0})

new_floor_cols = housing['所在楼层'].apply(extract_floor)
housing = pd.concat([housing, new_floor_cols], axis=1)
housing = housing.drop('所在楼层', axis=1)


In [7]:
multi_label_explosion(housing,'房屋朝向',' ')
housing = housing.drop('房屋朝向', axis=1)


检测到的基础房屋朝向类型共有 8 种：
['南' '北' '东' '西' '西北' '西南' '东南' '东北']


In [9]:
from utils import process_num

housing['建筑面积'] = housing['建筑面积'].apply(process_num)

In [10]:
housing.columns

Index(['城市', '区域', '板块', 'price', 'unit_price', '建筑面积', '建筑结构', '装修情况', '梯户比例',
       '配备电梯', '别墅类型', '挂牌时间', '交易权属', '上次交易', '房屋用途', '房屋年限', '产权所属', '房源标签',
       '核心卖点', '户型介绍', '周边配套', '交通出行', 'x', 'y', '年份', 'nearest_park_dist_km',
       'nearest_park_index', 'nearest_park_industry', 'log_price',
       'log_unit_price', 'area', 'distance_to_center', 'rooms', 'halls',
       'kitchens', 'bathrooms', '看房时间_is_只周末可看', '看房时间_is_下班后可看',
       '看房时间_is_提前预约随时可看', '看房时间_is_有租户需预约', 'floor_level', 'total_floors',
       '房屋朝向_is_南', '房屋朝向_is_北', '房屋朝向_is_东', '房屋朝向_is_西', '房屋朝向_is_西北',
       '房屋朝向_is_西南', '房屋朝向_is_东南', '房屋朝向_is_东北'],
      dtype='object')

In [ ]:
housing['建筑结构'].value_counts()

建筑面积
57.20     164
58.10     163
57.90     156
58.00     145
57.40     139
         ... 
286.33      1
235.47      1
372.29      1
291.51      1
11.30       1
Name: count, Length: 21777, dtype: int64